# 7A · Your First Optimiser — Where Should the Money Go?
### Financial Analytics — Module 7

Prescriptive analytics answers **"what should we do?"** — and its purest form is **optimisation**: given an objective and constraints, find the best possible decision, provably.

The scenario: you are MoneyMart's treasurer. **₹100 crore** of surplus cash must be parked across four instruments for the quarter:

| Instrument | Annual yield | Liquidity | Risk note |
|---|---|---|---|
| Overnight fund | 6.2% | Same-day | Safest |
| Treasury bills (91d) | 6.8% | 3 days | Sovereign |
| Bank fixed deposit | 7.1% | Penalty to break | Bank risk |
| Corporate deposit | 8.4% | Locked | Credit risk |

Maximise yield. But the board imposes rules — and the rules are where the realism lives.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

instruments = ["Overnight", "T-bills", "Bank FD", "Corp deposit"]
yields = np.array([0.062, 0.068, 0.071, 0.084])
TOTAL = 100.0    # Rs crore

---
## 1. The anatomy of every optimisation

Every prescriptive problem — this one, a portfolio, an airline's pricing — has exactly three parts. Name them before touching any solver:

1. **Decision variables** — the numbers we get to choose: `x = [x1, x2, x3, x4]`, crores in each instrument
2. **Objective** — the thing to maximise: total yield = `0.062·x1 + 0.068·x2 + 0.071·x3 + 0.084·x4`
3. **Constraints** — the rules that make it a real decision, not arithmetic:
   - All money placed: `x1 + x2 + x3 + x4 = 100`
   - Liquidity: at least **₹30 cr** in same-day + 3-day instruments: `x1 + x2 ≥ 30`
   - Credit-risk cap: corporate deposits at most **₹20 cr**: `x4 ≤ 20`
   - Single-bank cap: FD at most **₹35 cr**: `x3 ≤ 35`
   - No shorting cash: every `x ≥ 0`

*(Without constraints, the "answer" is trivial and useless: 100% into the highest yield. Constraints are not obstacles to the decision — they ARE the decision problem.)*

In [ ]:
# scipy's linprog MINIMISES, so we minimise the NEGATIVE yield (same thing, flipped sign)
c_obj = -yields

# Inequalities must be written as  A_ub @ x <= b_ub  - so ">= 30" becomes "-(x1+x2) <= -30"
A_ub = [[-1, -1,  0,  0],    # liquidity:  -(x1+x2) <= -30
        [ 0,  0,  0,  1],    # corp cap:    x4 <= 20
        [ 0,  0,  1,  0]]    # FD cap:      x3 <= 35
b_ub = [-30, 20, 35]

A_eq = [[1, 1, 1, 1]]        # all money placed
b_eq = [TOTAL]

res = linprog(c_obj, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
              bounds=[(0, None)]*4, method="highs")

alloc = pd.Series(res.x, index=instruments)
print(alloc.round(1).to_string())
print(f"\nOptimal annual yield: {-res.fun/TOTAL*100:.3f}%   (Rs {-res.fun:.2f} cr per year)")

---
## 2. Read the answer like an analyst

The optimiser didn't just give numbers — it revealed **the logic of the problem**:

- **Corp deposit is filled to its cap (20)** — highest yield, so the solver buys every crore the risk rule allows
- **Bank FD is filled to its cap (35)** — next-best yield, same story
- **T-bills take the rest of the liquidity duty (30... check it)** and **Overnight gets only what's forced**

Every rupee sits where it does *because of a specific constraint*. When an allocation is optimal, the interesting question flips from "what's the answer?" to **"which constraints are doing the work?"** — the *binding* constraints (satisfied with equality). Relax a binding constraint and the answer improves; relax a slack one and nothing changes.

In [ ]:
# Which constraints bind? Check each with the solution in hand:
x = res.x
checks = {
    "Liquidity  x1+x2 >= 30": (x[0]+x[1], 30, x[0]+x[1] <= 30 + 1e-6),
    "Corp cap   x4 <= 20   ": (x[3], 20, x[3] >= 20 - 1e-6),
    "FD cap     x3 <= 35   ": (x[2], 35, x[2] >= 35 - 1e-6),
}
for name, (val, lim, binding) in checks.items():
    print(f"{name}  value {val:5.1f} vs limit {lim}   {'BINDING' if binding else 'slack'}")

---
## 3. Shadow prices: what is a rule COSTING us?

The most valuable output of any optimisation is not the allocation — it's the **price of each rule**. If the board relaxed the corporate-deposit cap by ₹1 crore, how much extra yield would we earn? That number is the constraint's **shadow price**, and we can measure it the honest, brute-force way: perturb and re-solve.

In [ ]:
def solve(liq=30, corp_cap=20, fd_cap=35):
    r = linprog(-yields, A_ub=[[-1,-1,0,0],[0,0,0,1],[0,0,1,0]],
                b_ub=[-liq, corp_cap, fd_cap],
                A_eq=[[1,1,1,1]], b_eq=[TOTAL], bounds=[(0,None)]*4, method="highs")
    return -r.fun

base = solve()
print(f"{'Rule relaxed by Rs 1 cr':<32}{'extra yield (Rs cr/yr)':>24}")
print(f"{'Corp cap 20 -> 21':<32}{solve(corp_cap=21)-base:>24.4f}")
print(f"{'FD cap   35 -> 36':<32}{solve(fd_cap=36)-base:>24.4f}")
print(f"{'Liquidity 30 -> 29 (eased)':<32}{solve(liq=29)-base:>24.4f}")

Read those numbers as *prices*: each says what one crore of regulatory headroom is worth per year. Now the treasurer can walk into the board meeting with a genuinely prescriptive sentence:

> *"Our liquidity rule costs us ₹X lakh a year versus the corp-cap rule's ₹Y — if we're revisiting one policy, here's the order."*

That sentence — **pricing the rules, not just obeying them** — is the difference between running an optimiser and doing prescriptive analytics.

### ✏️ Exercise 1
The board adds a rule: Overnight must hold at least ₹10 cr (operational buffer). Add the constraint, re-solve. What does the buffer *cost* per year, and which instrument paid for it?

### ✏️ Exercise 2
Rates move: corp deposits fall to 7.0%. Re-solve. Does the corp cap still bind? What's the general lesson about how sensitive optimal *allocations* are to input estimates? (Hint: allocations often jump discontinuously while the optimal *value* moves smoothly — optimisers are drama queens about inputs. Remember this in Module 12.)

In [ ]:
# your code here


---
## 4. The warning label on every optimiser

An optimiser doesn't just use your inputs — it **exploits** them. If one yield estimate is 0.3% too high by error, the solver will *pile money onto the error*, because to the solver an error and an opportunity look identical. Garbage in → confident, maximised garbage out.

This is why prescriptive sits at the TOP of the ladder: it consumes everything below. Bad descriptive data (Module 1/3), a misdiagnosed driver (Module 5), an overconfident forecast (Module 6) — the optimiser amplifies them all into a decisive, wrong action. The four rungs are not a menu; they are a **load-bearing stack**.

Defences, in order: (1) sensitivity analysis — always ask how the answer moves when inputs wiggle (you just did); (2) constraints as humility — caps and floors limit how hard the solver can lean on any single estimate; (3) Module 10's move — replace point inputs with distributions and optimise across scenarios.

---
*AI disclosure: ______*

In [ ]:
# workspace
